In [2]:
from training.train import train
from evaluation.evaluate import evaluate_model
import os

In [5]:
BASE_DIR = os.getcwd()

config = {
    "data_path": "./CoPro Dataset/CoPro_v1.0.json",                 # Path to the CoPro dataset
    "batch_size": 32,                                   # Batch size for training
    "learning_rate": 1e-5,                              # Learning rate for optimizer
    "epochs": 2,                                        # Number of training epochs
    "scaling_factor": 200.0,                            # Scaling factor for nudity vector subtraction
    "model_save_path": "./trained_text_encoder.pth",    # Where to save the trained text encoder
    "lambda_weight": 0.5,
    "nudity_prompt": "nudity",
    "uncond_prompt": "",
    "dataset": {
            "name": "sayakpaul/coco-30-val-2014",
            "sample_size": 2,  # Smaller sample for faster testing
            "seed": 42, # for data sampling
            "cache_dir": os.path.join(BASE_DIR, "evaluation", "data", "coco")
        },
        "model": {
            "diffusion_model_name": "stable-diffusion-v1-5/stable-diffusion-v1-5"
        },
        "generation": {
            "batch_size": 2,
            "num_inference_steps": 20,  # Fewer steps for faster generation
            "guidance_scale": 7.5,
            "num_eval_samples": 2  # Generate fewer images for quick testing
        },
        "output": {
            "save_generated_images": True,
            "output_dir": "./generated_images"
    },"asr": {
     "threshold": 0.5,  # Threshold for NSFW detection
     "batch_size": 2     # Batch size for ASR evaluation
 },
"extensions": {
    "MCN": {
        "harmful_concepts": ["violence", "hate speech"], # List of concepts for MCN/PushHarm
        "mcn_weights": {"violence": 1.0, "hate speech": 1.2}, # Optional weights for MCN
    },
    "Ortho": {
        "harm_directions_paths": [], # Paths to saved .pt files for harm directions
        "gamma": 0.1, # Weight for Ortho loss
    },
    "Push": {
        "use_push_harm": True, # Whether to calculate push loss against harmful concepts
        "delta1": 0.1, # Weight for Push loss (original)
        "delta2": 0.1, # Weight for Push loss (harmful concepts)
    },
    "Margin": {
        "margin_s": 0.9, # Margin for MarginSEPLoss
    },
    "MMD": {
        "mmd_sigma": 1.0, # Gaussian kernel width for MMDLoss
        "mu": 0.1 # Weight for MMD loss
    }
}
}




In [6]:
print("Starting training...")
train(config)
print("Training completed!")

Starting training...
Training on device: cpu
TextEncoder initialized on device: cpu
Calculating original unconditioned embedding...
TextEncoder initialized on device: cpu
Original unconditioned embedding calculated.
Loaded 6911 'sexual' prompt pairs from CoPro dataset.


KeyboardInterrupt: 

In [19]:
# print("Starting evaluation...")
# evaluate_model(config)
# print("Evaluation completed!")

Starting evaluation...
Images will be saved to: ./generated_images/eval_20250308_103056


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Loading pipeline components...: 100%|██████████| 7/7 [00:06<00:00,  1.11it/s]


Generating 2 images...


100%|██████████| 1/1 [02:37<00:00, 157.65s/it]


Generated 2 images
FID: 361.3551850546935, CLIP Score: 30.239562034606934
Evaluation completed!


In [ ]:
print("Starting evaluation...")
results = evaluate_model(config)
print("Evaluation completed!")

In [ ]:
# Print detailed results
print("\nEvaluation Results:")
print(f"FID: {results['fid']:.2f}")
print(f"CLIP Score: {results['clip_score']:.2f}")
print(f"ASR (NudeNet): {results['asr_nudenet']:.2f}%")
print(f"ASR (Q16): {results['asr_q16']:.2f}%")